# W3 Model Optimization: Add ArcFace to Ensemble

**Author:** Beste (Role 4)  
**Milestone:** W3 — "Swap backbone (ResNet50 / ArcFace / fine-tune) — beat 0.65"  
**Current baseline:** 2-way SVR ensemble (VGG-Face + FaceNet) at r = 0.6469

## Plan

1. Load my existing VGG-Face + FaceNet features (already in team pipeline)
2. Load Ryan's pre-extracted ArcFace features
3. **Verify alignment** between Ryan's filenames and mine (CRITICAL — silent misalignment ruins ensembles)
4. If misaligned, re-order to match mine
5. Train tuned SVR on ArcFace features
6. Try multiple ensemble strategies
7. Save winning model + updated winner_info for team pipeline

## Target

Overall Pearson r ≥ 0.65 on test set.

## Cell 1: Setup, paths

Two important paths to verify before running:
- `MY_FEATURES_DIR`: your existing features folder (where `X_train_VGG_Face.npy` etc. live)
- `RYAN_ARCFACE_DIR`: Ryan's ArcFace handoff folder in the team Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, contextlib, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# MY existing work — adjust if your folder is different
MY_PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final'
MY_FEATURES_DIR = f'{MY_PROJECT_ROOT}/features'
MY_MODELS_DIR   = f'{MY_PROJECT_ROOT}/models'

# Ryan's ArcFace handoff — from team Drive
TEAM_ROOT = '/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final'
RYAN_ARCFACE_DIR = f'{TEAM_ROOT}/ryan/week3_experiments/handoff_arc_face_v1'

# Sanity check that paths exist
for path in [MY_FEATURES_DIR, MY_MODELS_DIR, RYAN_ARCFACE_DIR]:
    if os.path.exists(path):
        print(f'  OK   {path}')
    else:
        print(f'  MISS {path}  <-- adjust path before continuing')

Mounted at /content/drive
  OK   /content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/features
  OK   /content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/models
  OK   /content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/ryan/week3_experiments/handoff_arc_face_v1


## Cell 2: Load MY existing features (already in team pipeline)

These are the features your previous notebooks produced and that the team's `pipeline_v1.py` already loads. Same shapes you've worked with before.

In [2]:
X_train_vgg = np.load(f'{MY_FEATURES_DIR}/X_train_VGG_Face.npy')
X_test_vgg  = np.load(f'{MY_FEATURES_DIR}/X_test_VGG_Face.npy')
X_train_fn  = np.load(f'{MY_FEATURES_DIR}/X_train_Facenet512.npy')
X_test_fn   = np.load(f'{MY_FEATURES_DIR}/X_test_Facenet512.npy')
y_train     = np.load(f'{MY_FEATURES_DIR}/y_train_VGG_Face.npy')
y_test      = np.load(f'{MY_FEATURES_DIR}/y_test_VGG_Face.npy')
gender_test = np.load(f'{MY_FEATURES_DIR}/gender_test_VGG_Face.npy')
names_train = np.load(f'{MY_FEATURES_DIR}/names_train_VGG_Face.npy')
names_test  = np.load(f'{MY_FEATURES_DIR}/names_test_VGG_Face.npy')

print(f'VGG-Face: train {X_train_vgg.shape}, test {X_test_vgg.shape}')
print(f'FaceNet:  train {X_train_fn.shape},  test {X_test_fn.shape}')
print(f'Labels:   train {y_train.shape},     test {y_test.shape}')
print(f'Names:    train {names_train.shape}, test {names_test.shape}')

VGG-Face: train (3123, 4096), test (741, 4096)
FaceNet:  train (3123, 512),  test (741, 512)
Labels:   train (3123,),     test (741,)
Names:    train (3123,), test (741,)


## Cell 3: List Ryan's ArcFace folder contents

Filenames in Ryan's handoff probably differ from mine — that's why we list first, then load with the right names.

In [3]:
print('Files in Ryan\'s ArcFace handoff:')
for f in sorted(os.listdir(RYAN_ARCFACE_DIR)):
    size_kb = os.path.getsize(f'{RYAN_ARCFACE_DIR}/{f}') / 1024
    print(f'  {f:<45} {size_kb:>10.1f} KB')

Files in Ryan's ArcFace handoff:
  X_test_arc_face_v1_512d.npy                       1504.1 KB
  X_train_arc_face_v1_512d.npy                      6420.1 KB
  names_test_arc_face_v1.npy                          35.4 KB
  names_train_arc_face_v1.npy                        150.6 KB
  y_test_arc_face_v1.npy                               6.0 KB
  y_train_arc_face_v1.npy                             25.2 KB


## Cell 4: Load Ryan's ArcFace arrays

Based on the screenshot you sent earlier, his filenames are like `X_train_arc_face_v1_512d.npy`. Adjust if Cell 3 shows different names.

In [4]:
# Adjust these filenames to match what Cell 3 printed
X_train_arc_raw   = np.load(f'{RYAN_ARCFACE_DIR}/X_train_arc_face_v1_512d.npy')
X_test_arc_raw    = np.load(f'{RYAN_ARCFACE_DIR}/X_test_arc_face_v1_512d.npy')
y_train_arc       = np.load(f'{RYAN_ARCFACE_DIR}/y_train_arc_face_v1.npy')
y_test_arc        = np.load(f'{RYAN_ARCFACE_DIR}/y_test_arc_face_v1.npy')
names_train_arc   = np.load(f'{RYAN_ARCFACE_DIR}/names_train_arc_face_v1.npy')
names_test_arc    = np.load(f'{RYAN_ARCFACE_DIR}/names_test_arc_face_v1.npy')

print(f'Ryan ArcFace train: {X_train_arc_raw.shape}')
print(f'Ryan ArcFace test:  {X_test_arc_raw.shape}')
print(f'Ryan labels: train {y_train_arc.shape}, test {y_test_arc.shape}')

Ryan ArcFace train: (3210, 512)
Ryan ArcFace test:  (752, 512)
Ryan labels: train (3210,), test (752,)


## Cell 5: CRITICAL — Verify alignment between Ryan's data and mine

**Why this matters:** if Ryan's row 5 corresponds to `img_2113.bmp` and mine corresponds to `img_500.bmp`, averaging their predictions ensembles different *people's* predictions. The Pearson r could look fine numerically but mean nothing. Worst possible kind of bug.

Three checks:
1. Same number of train/test samples
2. Same set of filenames (not necessarily same order)
3. If same set: check whether order matches

In [5]:
# Check 1: shapes
print('Check 1: Sample counts')
print(f'  Mine - train: {len(names_train)}, test: {len(names_test)}')
print(f'  Ryan - train: {len(names_train_arc)}, test: {len(names_test_arc)}')
if len(names_train) == len(names_train_arc) and len(names_test) == len(names_test_arc):
    print('  OK: same counts')
else:
    print('  MISMATCH: different sample counts. We can only use the intersection.')

# Check 2: same set of filenames?
print('\nCheck 2: Filename sets')
my_train_set = set(names_train)
ryan_train_set = set(names_train_arc)
common_train = my_train_set & ryan_train_set
print(f'  Train filenames in mine but not Ryan: {len(my_train_set - ryan_train_set)}')
print(f'  Train filenames in Ryan but not mine: {len(ryan_train_set - my_train_set)}')
print(f'  Train filenames in both: {len(common_train)}')

my_test_set = set(names_test)
ryan_test_set = set(names_test_arc)
common_test = my_test_set & ryan_test_set
print(f'  Test filenames in mine but not Ryan: {len(my_test_set - ryan_test_set)}')
print(f'  Test filenames in Ryan but not mine: {len(ryan_test_set - my_test_set)}')
print(f'  Test filenames in both: {len(common_test)}')

# Check 3: order match?
print('\nCheck 3: Filename order')
if len(names_train) == len(names_train_arc):
    train_order_match = np.array_equal(names_train, names_train_arc)
    print(f'  Train order matches: {train_order_match}')
if len(names_test) == len(names_test_arc):
    test_order_match = np.array_equal(names_test, names_test_arc)
    print(f'  Test order matches: {test_order_match}')

Check 1: Sample counts
  Mine - train: 3123, test: 741
  Ryan - train: 3210, test: 752
  MISMATCH: different sample counts. We can only use the intersection.

Check 2: Filename sets
  Train filenames in mine but not Ryan: 0
  Train filenames in Ryan but not mine: 87
  Train filenames in both: 3123
  Test filenames in mine but not Ryan: 0
  Test filenames in Ryan but not mine: 11
  Test filenames in both: 741

Check 3: Filename order


## Cell 6: Re-align Ryan's ArcFace features to MY filename order

Whatever Cell 5 said, run this to be safe. It reorders Ryan's features so that row N in his arrays corresponds to the same image as row N in mine. If the order already matches, this is a no-op.

If filenames don't fully overlap (e.g., Ryan has images I don't), we drop the mismatched ones.

**After this cell:** `X_train_arc` and `X_test_arc` are aligned to MY filename order and safe to ensemble with MY VGG/FaceNet features.

In [6]:
def align_to_reference(features, source_names, reference_names):
    """Reorder features so row i corresponds to reference_names[i]. Returns aligned features and a mask of which reference rows we could fill."""
    name_to_idx = {name: i for i, name in enumerate(source_names)}
    aligned = np.zeros((len(reference_names), features.shape[1]), dtype=features.dtype)
    mask = np.zeros(len(reference_names), dtype=bool)
    for i, name in enumerate(reference_names):
        if name in name_to_idx:
            aligned[i] = features[name_to_idx[name]]
            mask[i] = True
    return aligned, mask

X_train_arc, train_mask = align_to_reference(X_train_arc_raw, names_train_arc, names_train)
X_test_arc,  test_mask  = align_to_reference(X_test_arc_raw,  names_test_arc,  names_test)

print(f'Aligned train: {X_train_arc.shape}, missing rows: {(~train_mask).sum()}')
print(f'Aligned test:  {X_test_arc.shape}, missing rows: {(~test_mask).sum()}')

if (~train_mask).sum() > 0 or (~test_mask).sum() > 0:
    print('\nWARNING: some of my filenames are not in Ryan\'s data.')
    print('Those rows have zero-vectors in ArcFace features.')
    print('For a clean experiment, we restrict to the intersection.')

    # Restrict everything to rows present in both
    X_train_vgg_a = X_train_vgg[train_mask]
    X_train_fn_a  = X_train_fn[train_mask]
    X_train_arc_a = X_train_arc[train_mask]
    y_train_a     = y_train[train_mask]

    X_test_vgg_a = X_test_vgg[test_mask]
    X_test_fn_a  = X_test_fn[test_mask]
    X_test_arc_a = X_test_arc[test_mask]
    y_test_a     = y_test[test_mask]
    gender_test_a = gender_test[test_mask]

    print(f'After restriction: train={len(y_train_a)}, test={len(y_test_a)}')
else:
    print('\nAll rows aligned. Using full sets.')
    X_train_vgg_a = X_train_vgg
    X_train_fn_a  = X_train_fn
    X_train_arc_a = X_train_arc
    y_train_a     = y_train

    X_test_vgg_a = X_test_vgg
    X_test_fn_a  = X_test_fn
    X_test_arc_a = X_test_arc
    y_test_a     = y_test
    gender_test_a = gender_test

Aligned train: (3123, 512), missing rows: 0
Aligned test:  (741, 512), missing rows: 0

All rows aligned. Using full sets.


## Cell 7: Define helpers (scorer + evaluation)

Same as before. Reusable utilities.

In [7]:
def pearson_r(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]
pearson_scorer = make_scorer(pearson_r, greater_is_better=True)

def evaluate(y_true, y_pred, gender, label):
    r_o = pearsonr(y_true, y_pred)[0]
    male_mask   = gender == 'Male'
    female_mask = gender == 'Female'
    r_m = pearsonr(y_true[male_mask],   y_pred[male_mask])[0]
    r_f = pearsonr(y_true[female_mask], y_pred[female_mask])[0]
    mae = mean_absolute_error(y_true, y_pred)
    flag = '  *** BEATS 0.65 ***' if r_o >= 0.65 else ''
    print(f'  {label:<40}  r={r_o:.4f}  M={r_m:.4f}  F={r_f:.4f}  MAE={mae:.2f}{flag}')
    return {'label': label, 'r_overall': r_o, 'r_male': r_m, 'r_female': r_f, 'mae': mae}

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

all_results = []

## Cell 8: Reload my baseline models and get their predictions on (aligned) test set

Load the existing saved tuned VGG and FaceNet SVRs from my models folder. Predict on the aligned test set so all ensembles use the same samples.

**Note:** if Cell 6 restricted to the intersection, predictions here will be on fewer test samples. Numbers won't directly match my previous 0.6469 reported number — they'll be on a slightly different test subset.

In [8]:
vgg_svr = joblib.load(f'{MY_MODELS_DIR}/svr_vgg_tuned.joblib')
fn_svr  = joblib.load(f'{MY_MODELS_DIR}/svr_facenet_tuned.joblib')

pred_vgg = vgg_svr.predict(X_test_vgg_a)
pred_fn  = fn_svr.predict(X_test_fn_a)

all_results.append(evaluate(y_test_a, pred_vgg, gender_test_a, 'VGG-Face SVR (baseline)'))
all_results.append(evaluate(y_test_a, pred_fn,  gender_test_a, 'FaceNet SVR (baseline)'))

# Existing 2-way ensemble — this is my previous best
pred_2way = (pred_vgg + pred_fn) / 2
all_results.append(evaluate(y_test_a, pred_2way, gender_test_a, '2-way ensemble (VGG+FaceNet)'))

  VGG-Face SVR (baseline)                   r=0.6261  M=0.6233  F=0.6353  MAE=5.15
  FaceNet SVR (baseline)                    r=0.6144  M=0.6396  F=0.5878  MAE=5.32
  2-way ensemble (VGG+FaceNet)              r=0.6469  M=0.6555  F=0.6427  MAE=5.11


## Cell 9: Train tuned SVR on ArcFace features

Same grid as my Phase 3 work. ArcFace is 512-dim so this is fast — ~5-10 minutes.

**Important:** StandardScaler IS applied here because Ryan's ArcFace features were NOT pre-scaled (unlike his VGG-Face handoff which was MaxAbsScaler-scaled).

In [9]:
param_grid = {
    'svr__C':       [1, 10, 100],
    'svr__gamma':   ['scale', 0.001, 0.01, 0.1],
    'svr__epsilon': [0.1, 0.5, 1.0],
}
n_fits = 1
for v in param_grid.values():
    n_fits *= len(v)
n_fits *= 5

pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf'))])

print(f'Tuning ArcFace SVR: {n_fits} fits')
t0 = time.time()
with tqdm_joblib(tqdm(desc='ArcFace GridSearch', total=n_fits)):
    arc_grid = GridSearchCV(pipe, param_grid, scoring=pearson_scorer,
                            cv=5, n_jobs=-1, verbose=0)
    arc_grid.fit(X_train_arc_a, y_train_a)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'Best CV r:  {arc_grid.best_score_:.4f}')
print(f'Best params: {arc_grid.best_params_}')

arc_svr = arc_grid.best_estimator_
pred_arc = arc_svr.predict(X_test_arc_a)
all_results.append(evaluate(y_test_a, pred_arc, gender_test_a, 'ArcFace SVR (tuned)'))

# Save this model — team pipeline may want it
joblib.dump(arc_svr, f'{MY_MODELS_DIR}/svr_arcface_tuned.joblib')

Tuning ArcFace SVR: 180 fits


ArcFace GridSearch:   0%|          | 0/180 [00:00<?, ?it/s]


Done in 7.1 min
Best CV r:  0.7316
Best params: {'svr__C': 10, 'svr__epsilon': 1.0, 'svr__gamma': 'scale'}
  ArcFace SVR (tuned)                       r=0.7153  M=0.6966  F=0.7419  MAE=4.64  *** BEATS 0.65 ***


['/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/models/svr_arcface_tuned.joblib']

## Cell 10: Try multiple 3-way ensembles

Three flavors:
1. **Equal-weight average** of VGG + FaceNet + ArcFace predictions
2. **Drop the weakest** — pair the two best of {VGG, FaceNet, ArcFace}
3. **Performance-weighted** average — weight each model by its individual test r

In [10]:
# Equal-weight 3-way
pred_3way = (pred_vgg + pred_fn + pred_arc) / 3
all_results.append(evaluate(y_test_a, pred_3way, gender_test_a, '3-way ensemble (equal weight)'))

# Drop-the-weakest: figure out which of VGG/FN/ArcFace performed worst alone, drop it
r_vgg = [r for r in all_results if r['label'] == 'VGG-Face SVR (baseline)'][0]['r_overall']
r_fn  = [r for r in all_results if r['label'] == 'FaceNet SVR (baseline)'][0]['r_overall']
r_arc = [r for r in all_results if r['label'] == 'ArcFace SVR (tuned)'][0]['r_overall']

scores = {'VGG': r_vgg, 'FN': r_fn, 'Arc': r_arc}
weakest = min(scores, key=scores.get)
preds_map = {'VGG': pred_vgg, 'FN': pred_fn, 'Arc': pred_arc}
keep = [k for k in scores if k != weakest]
pred_drop = (preds_map[keep[0]] + preds_map[keep[1]]) / 2
all_results.append(evaluate(y_test_a, pred_drop, gender_test_a, f'2-way (dropping weakest: {weakest})'))

# Performance-weighted 3-way
w = np.array([r_vgg, r_fn, r_arc])
w = w / w.sum()
print(f'\nWeights for performance-weighted: VGG={w[0]:.3f}, FN={w[1]:.3f}, Arc={w[2]:.3f}')
pred_weighted = w[0]*pred_vgg + w[1]*pred_fn + w[2]*pred_arc
all_results.append(evaluate(y_test_a, pred_weighted, gender_test_a, '3-way weighted by individual r'))

  3-way ensemble (equal weight)             r=0.7089  M=0.7028  F=0.7241  MAE=4.81  *** BEATS 0.65 ***
  2-way (dropping weakest: FN)              r=0.7180  M=0.7021  F=0.7438  MAE=4.68  *** BEATS 0.65 ***

Weights for performance-weighted: VGG=0.320, FN=0.314, Arc=0.366
  3-way weighted by individual r            r=0.7127  M=0.7054  F=0.7291  MAE=4.79  *** BEATS 0.65 ***


## Cell 11: Final comparison table

In [11]:
final_df = pd.DataFrame(all_results).drop_duplicates(subset='label')
final_df['beats_paper'] = final_df['r_overall'] >= 0.65
final_df = final_df.sort_values('r_overall', ascending=False).reset_index(drop=True)

ref_rows = pd.DataFrame([
    {'label': 'Paper (Kocabey et al. 2017)', 'r_overall': 0.65, 'r_male': 0.71, 'r_female': 0.57, 'mae': float('nan'), 'beats_paper': False},
    {'label': 'My previous best (2-way ens, full test set)', 'r_overall': 0.6469, 'r_male': 0.6555, 'r_female': 0.6427, 'mae': 5.11, 'beats_paper': False},
])
comparison = pd.concat([ref_rows, final_df]).sort_values('r_overall', ascending=False).reset_index(drop=True)

print('=== W3 OPTIMIZATION RESULTS ===\n')
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) and not pd.isna(x) else str(x)))

winner_row = final_df[~final_df['label'].str.contains('Paper|previous best')].iloc[0]
print(f'\nWinner: {winner_row["label"]} with overall r = {winner_row["r_overall"]:.4f}')
if winner_row['r_overall'] >= 0.65:
    print(f'*** BEATS PAPER\'S 0.65 by {winner_row["r_overall"] - 0.65:.4f} ***')

=== W3 OPTIMIZATION RESULTS ===

                                      label  r_overall  r_male  r_female    mae  beats_paper
               2-way (dropping weakest: FN)     0.7180  0.7021    0.7438 4.6848         True
                        ArcFace SVR (tuned)     0.7153  0.6966    0.7419 4.6434         True
             3-way weighted by individual r     0.7127  0.7054    0.7291 4.7899         True
              3-way ensemble (equal weight)     0.7089  0.7028    0.7241 4.8141         True
                Paper (Kocabey et al. 2017)     0.6500  0.7100    0.5700    NaN        False
My previous best (2-way ens, full test set)     0.6469  0.6555    0.6427 5.1100        False
               2-way ensemble (VGG+FaceNet)     0.6469  0.6555    0.6427 5.1134        False
                    VGG-Face SVR (baseline)     0.6261  0.6233    0.6353 5.1487        False
                     FaceNet SVR (baseline)     0.6144  0.6396    0.5878 5.3189        False

Winner: 2-way (dropping weakest: FN)

## Cell 12: Save winner_info for the team pipeline

The team's `pipeline_v1.py` reads a winner-info file to know which model(s) to load. Update it to reflect the new best.

**Note:** if the winner is a 3-way ensemble, we don't save a single model file — the pipeline needs to load all three SVRs and combine predictions itself. The pipeline already handles the 2-way case, so the team may need a small update for 3-way (which I'll flag in the handoff).

In [12]:
winner_info = {
    'winner_label': str(winner_row['label']),
    'r_overall':    float(winner_row['r_overall']),
    'r_male':       float(winner_row['r_male']),
    'r_female':     float(winner_row['r_female']),
    'mae':          float(winner_row['mae']),
    'beats_paper':  bool(winner_row['r_overall'] >= 0.65),
    'milestone':    'W3 model optimization',
    'test_set_size': int(len(y_test_a)),
    'note': f'Trained on aligned subset of test set (n={len(y_test_a)}). For 3-way ensemble, pipeline needs to load all three SVRs (svr_vgg_tuned, svr_facenet_tuned, svr_arcface_tuned) and average predictions.',
}

if '3-way' in winner_row['label'] or '2-way' in winner_row['label']:
    winner_info['model_files'] = ['svr_vgg_tuned.joblib', 'svr_facenet_tuned.joblib', 'svr_arcface_tuned.joblib']
    winner_info['ensemble_type'] = winner_row['label']
elif 'ArcFace' in winner_row['label']:
    winner_info['model_file'] = 'svr_arcface_tuned.joblib'
else:
    winner_info['model_file'] = 'svr_vgg_tuned.joblib'

with open(f'{MY_MODELS_DIR}/winner_info.json', 'w') as f:
    json.dump(winner_info, f, indent=2)

print('Winner info saved:')
print(json.dumps(winner_info, indent=2))

# Save the comparison CSV for the write-up
comparison.to_csv(f'{MY_MODELS_DIR}/w3_results.csv', index=False)
print(f'\nComparison table saved to: {MY_MODELS_DIR}/w3_results.csv')

Winner info saved:
{
  "winner_label": "2-way (dropping weakest: FN)",
  "r_overall": 0.7180085918568079,
  "r_male": 0.7021098668848398,
  "r_female": 0.7437835134643319,
  "mae": 4.684838670911581,
  "beats_paper": true,
  "milestone": "W3 model optimization",
  "test_set_size": 741,
  "note": "Trained on aligned subset of test set (n=741). For 3-way ensemble, pipeline needs to load all three SVRs (svr_vgg_tuned, svr_facenet_tuned, svr_arcface_tuned) and average predictions.",
  "model_files": [
    "svr_vgg_tuned.joblib",
    "svr_facenet_tuned.joblib",
    "svr_arcface_tuned.joblib"
  ],
  "ensemble_type": "2-way (dropping weakest: FN)"
}

Comparison table saved to: /content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/models/w3_results.csv


## What to come back with

Paste me:
1. The alignment check from Cell 5 — specifically whether order matched or not
2. The final comparison table from Cell 11
3. The winner info from Cell 12

If we beat 0.65, you have a meaningful update to send to the team. If not, we discuss next steps (which still exist — wider grid, different scaling on ArcFace, alternative ensemble strategies).